## A scripted agent that asks about your day

This notebook runs a basic conversational agent that asks a user about their day, hour by hour. For each hour, the agent asks what the user did and then asks follow-up questions for the details that the user did not mention yet:

* **with_whom**: who took part
* **what**: what the activity was about, e.g. the food, objects or topics
* **with_what**: the tools, devices or vehicles that were used
* **where**: the place
* **when**: the time at which it started
* **how_long**: how long it took

The dialogue itself is scripted: the agent always asks the same questions in the same order. An LLM served by a local [llama.cpp](https://github.com/ggml-org/llama.cpp) server is only used to interpret the answers of the user: it decides which details the user already gave and fills the slots of the event. After each event, the agent moves on to the next hour after the event ended, until the user types ```stop``` or the day is over.

The result is saved as a JSON file with the same structure as the examples in ```data/example_days```: an object with ```event1```, ```event2```, ... and the ```coreference_chains``` of the people in the events.

The code of the agent is in ```day_interviewer.py``` in the same folder as this notebook.

## Prerequisites

The agent calls a llama_cpp server that needs to run locally (https://llama-cpp.com).
The installation procedure is described in [llama.cpp](https://github.com/ggml-org/llama.cpp) (e.g. ```brew install llama.cpp``` on macOS).


### Starting the llama.cpp server

After installing llama_cpp, open a terminal launch the server with the Qwen2.7-7B model as follows:

```bash
llama-server -hf Qwen/Qwen2.5-7B-Instruct-GGUF:Q4_K_M --port 8080 -c 4096
```

This will load the Qen model from Huggingface.co. Other models that you can server through llama_cpp can be found here: [huggingface.co](https://huggingface.co/models?library=gguf). The server then provides an OpenAI-compatible API at ```http://localhost:8080/v1```. No API key is needed, but the ```openai``` client requires a value, so any string will do.


### Client modules to make calls to the server

The agent uses the ```openai``` Python client to call the llama.cpp server, ```ipywidgets``` for the chat window and ```pandas``` to show the result:

```
!pip install openai
!pip install ipywidgets
!pip install pandas
```

These packages should have been installed already through the requirements.txt in your virtual environment.


In [1]:
import sys
from pathlib import Path

# Jupyter runs the notebook in its own folder. We add the root of the project to the path,
# so that the agent can be imported from src.cltl.demo_chatbot
PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "cltl").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json
import pandas as pd
from openai import OpenAI
from src.cltl.demo_chatbot.day_interviewer import DayInterviewer
from src.cltl.demo_chatbot.chat_window import ChatWindow
from src.cltl.demo_chatbot.day_statistics import get_turns

## Settings

* ```LLAMA_CPP_URL```: the address of the llama.cpp server
* ```EXAMPLES```: the folder with the example days. The agent takes the event types from these files and gives a few of the events to the LLM as examples of how to fill the slots.
* ```OUTPUT```: the folder in which the day is saved as ```day_<name>_<date>.json```, by default ```data/collected_days``` in the root of the project (not in the folder of this notebook). The file is saved after every event, so nothing is lost if the conversation is interrupted. The window shows where the file is saved. Note that the ```data``` folder is in ```.gitignore```, so your days are not committed to git and the folder may be shown in grey in your editor.
* ```FONT_SIZE```: the font size of the chat window in pixels at the start. It can also be changed with the slider at the top of the window.
* ```ASK_ALL```: with ```True```, the agent asks every question for every event, also when the user already gave the answer. With ```False```, the agent only asks for the details that are missing.

In [3]:
LLAMA_CPP_URL = "http://localhost:8080/v1"
EXAMPLES = PROJECT_ROOT / "data" / "example_days"
OUTPUT = PROJECT_ROOT / "data" / "collected_days"
FONT_SIZE = 18
ASK_ALL = False

## Connecting to the llama.cpp server

In [4]:
llama_client = OpenAI(base_url=LLAMA_CPP_URL, api_key="sk-no-key-required")

# llama.cpp serves the model it was started with, the name is only used for reporting
model = llama_client.models.list().data[0].id
print('Model served by llama.cpp:', model)

Model served by llama.cpp: Qwen/Qwen2.5-7B-Instruct-GGUF:Q4_K_M


## Talking to the agent

Run the next cell to open the chat window. If you changed ```day_interviewer.py``` or ```chat_window.py```, restart the kernel first, since Python only imports a module once. The questions of the agent and your answers are shown together in the window, with the agent on the left and your answers on the right. Type your answer in the text field below the conversation and press **enter** to send it. Each answer takes a few seconds, since the LLM needs to process it. While the agent is thinking, the text field is disabled.

* **Skip question**: sends an empty answer. When the agent asks what you did at a certain hour, this skips that hour. When the agent asks a follow-up question, e.g. who you were with, the question is left unanswered.
* **Font size**: use the slider at the top of the window to make the text bigger or smaller.
* **Stopping**: click the red **Stop** button or type ```stop```, ```bye```, ```done```, ```quit``` or ```that's all``` as an answer. If you stop in the middle of the follow-up questions of an hour, that event is saved with the answers you gave so far. The agent then groups the people in the coreference chains, saves the day and disables the window. The last message of the agent and the line below the window tell where the file is saved. The conversation also stops by itself when the day is over, after midnight.

The events are saved after each answered hour, so if the kernel is interrupted the events so far are kept, but the file then has no ```coreference_chains```. Always stop the conversation with **Stop** to get the complete file.

In [5]:
agent = DayInterviewer(llama_client, model, examples_folder=EXAMPLES, output_folder=OUTPUT, ask_all=ASK_ALL)
ChatWindow(agent, font_size=FONT_SIZE).show()

## The result

Run the next cells after the conversation is finished. The day is saved in the JSON file given at the end of the conversation. We load it again to show the events as a table. Each row is an event with the slots filled by the LLM. The ```when``` slot is not filled by the LLM but made from the start time, in the same way as in the example days, e.g. ```["05:45", "quarter to six", "6 o'clock", "early in the morning", "morning"]```.

In [6]:
if agent.path is None:
    raise ValueError("No events saved yet, first tell the agent about your day in the chat window.")
if not agent.done:
    print("The conversation is not finished yet, these are the events so far. Click Stop to finish it.")
print("The day is saved in", agent.path)
with open(agent.path) as json_file:
    day = json.load(json_file)

events = {key: event for key, event in day.items() if key.startswith("event")}
pd.DataFrame(events).T[["event_type", "event_label", "when", "how_long", "where", "what", "with_what", "with_whom", "description"]]

The conversation is not finished yet, these are the events so far. Click Stop to finish it.
The day is saved in /Users/piek/Desktop/Combot2026/ma-communicative-robots/data/collected_days/day_emma_20261021_1713.json


,event_type,event_label,when,how_long,where,what,with_what,with_whom,description
event1,personal_care,waking up,"[07:00, 7 o'clock, early in the morning, morning]",2,"[bedroom of her studio, bedroom, her studio, A...",[],[alarm clock],[],got out of bed after the alarm went off
event2,personal_care,showering and getting dressed,"[08:00, 8 o'clock, early in the morning, morning]",15,"[bathroom of her house, bathroom, her house, A...","[shower, clothes]",[bathroom],[],took a shower and got dressed
event3,meal,breakfast,"[09:00, 9 o'clock, in the morning, morning]",30,"[shared kitchen in the student flat, kitchen, ...","[yoghurt, fruits, coffee]","[bowl, spoon, mug]",[flatmate Lotte],ate yoghurt with fruits and drank coffee
event4,travel,cycling to Uni,"[10:00, 10 o'clock, in the morning, morning]",15,"[university, her university, Amsterdam, Nether...",[],[bicycle],[],cycled to the university


The questions of the agent about each event are kept as a list in ```agent_turn``` and the answers of the user as a list in ```human_turn```. Each turn has the utterance and the number of the turn in the conversation, e.g. ```{"turn_id": 9, "utterance": "What did you do at 7 o'clock?"}```. ```human_turn[i]``` is the answer to ```agent_turn[i]```, with the next ```turn_id```. A skipped question has an empty answer:

In [7]:
for key, event in events.items():
    print(f"{key}:")
    # days collected with an older version of the agent have no agent_turn
    questions = get_turns(event, "agent_turn")
    for i, answer in enumerate(get_turns(event, "human_turn")):
        if i < len(questions):
            print(f"    {questions[i]['turn_id']:>3} Agent:", questions[i]["utterance"])
        print(f"    {answer['turn_id']:>3} You:  ", answer["utterance"] or "(skipped)")

event1:
     11 Agent: What did you do at 7 o'clock?
     12 You:   My alarm went off and I got out of bed
     13 Agent: Who were you with?
     14 You:   I was alone
     15 Agent: What did you use for it, such as tools, devices or a vehicle?
     16 You:   (skipped)
     17 Agent: Where were you?
     18 You:   in my bedroom in my studio
     19 Agent: How long did it take?
     20 You:   2 minutes
event2:
     21 Agent: What did you do at 8 o'clock?
     22 You:   I had breakfast with my
     23 Agent: What did it involve? For example, what did you eat, make, read or talk about?
     24 You:   I took a shower and got dressed
     25 Agent: What did you use for it, such as tools, devices or a vehicle?
     26 You:   bathroom and clothes
     27 Agent: Where were you?
     28 You:   (skipped)
     29 Agent: How long did it take?
     30 You:   15 minutes
event3:
     31 Agent: What did you do at 9 o'clock?
     32 You:   I had breakfast with Lotte
     33 Agent: What did it involve? 

## Coreference chains

At the end of the conversation, the LLM groups the mentions of people in ```with_whom``` that refer to the same entity, e.g. ```his wife Gerda``` and ```Gerda```. Each entity has a chain with all its mentions in the events. The user is the narrator and is mentioned in ```who``` in every event.

In [8]:
for chain in day.get("coreference_chains", []):
    mentions = [f"{mention['event']}: {mention['mention']}" for mention in chain["mentions"]]
    print(f"{chain['entity_id']} ({chain['type']}, {chain['description']})")
    print("   ", "; ".join(mentions))

The full JSON of the first event, which can be compared with the events in ```data/example_days```:

In [9]:
print(json.dumps(day["event1"], indent=2, ensure_ascii=False))

{
  "event_type": "personal_care",
  "event_label": "waking up",
  "who": [
    "Emma"
  ],
  "what": [],
  "description": "got out of bed after the alarm went off",
  "agent_turn": [
    {
      "turn_id": 11,
      "utterance": "What did you do at 7 o'clock?"
    },
    {
      "turn_id": 13,
      "utterance": "Who were you with?"
    },
    {
      "turn_id": 15,
      "utterance": "What did you use for it, such as tools, devices or a vehicle?"
    },
    {
      "turn_id": 17,
      "utterance": "Where were you?"
    },
    {
      "turn_id": 19,
      "utterance": "How long did it take?"
    }
  ],
  "human_turn": [
    {
      "turn_id": 12,
      "utterance": "My alarm went off and I got out of bed"
    },
    {
      "turn_id": 14,
      "utterance": "I was alone"
    },
    {
      "turn_id": 16,
      "utterance": ""
    },
    {
      "turn_id": 18,
      "utterance": "in my bedroom in my studio"
    },
    {
      "turn_id": 20,
      "utterance": "2 minutes"
    }
  ],
  

## Using the agent in other code

The agent does not read or print anything itself: ```start()``` returns the first question and ```respond(utterance)``` takes the answer of the user and returns the next question. The conversation is finished when ```agent.done``` is ```True```. This makes it possible to connect the agent to other interfaces, such as a speech interface or the LEOLANI platform, e.g.:

```
agent = DayInterviewer(llama_client, model)
question = agent.start()
while not agent.done:
    answer = listen_to_the_user(question)
    question = agent.respond(answer)
```

The agent can also be run from a terminal in the root of the project:

```
python -m src.cltl.demo_chatbot.day_interviewer
```